In [ ]:
from datetime import datetime, timedelta, UTC

import numpy as np
import pandas as pd
import plotly.express as px

from aare.constants import TIME, TEMP
from aare_influx.field_request import FieldRequest
from aare_influx.remote_existenz_store import RemoteExistenzStore
from aare_train.preparation import resample
from aare_train.utils import join_many

# Forecasting hourly mean vs end of hour

I'm a big stupid dumdum, apparently. I thought I'd forecast the mean of every hour to get a more stable forecast,
but that doesn't align with the use case the model is intended for. This also applies to air temp, humidity, flow, etc.
For features with a sum, e.g. rainfall and sunshine duration, the sum per aggregate makes sense and should be kept.
For things like wind, it might be smart to have mean (for sustained) and max for peak.

How much different would it be to forecast to the end of the hour? Anything unexpected?

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("dulwich").setLevel(logging.WARNING)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
store = RemoteExistenzStore()

In [ ]:
tz = "Europe/Zurich"
now = datetime.now(UTC)
start = now - timedelta(weeks=10 * 52)
end = start + timedelta(weeks=52)

start = datetime(2004, 2, 1, 10, 3, 7, tzinfo=UTC)
end = now

loc = "bern"
target_mean = FieldRequest("hydro", "temperature", "1h", "mean", loc)
target_last = FieldRequest("hydro", "temperature", "1h", "last", loc)
target_first = FieldRequest("hydro", "temperature", "1h", "first", loc)

In [ ]:
df_mean = store.query((start, end), target_mean)
df_last = store.query((start, end), target_last)
df_first = store.query((start, end), target_first)
df = pd.merge(df_mean, df_last, on=TIME, suffixes=("_mean", "_last"))
df = pd.merge(df, df_first.rename(columns={"temperature_bern": "temperature_bern_first"}), on=TIME)
# df[TIME] = df[TIME].dt.tz_convert(tz)
df

In [ ]:
px.line(df, x=TIME, y=["temperature_bern_mean", "temperature_bern_first", "temperature_bern_last"])

In [ ]:
def get_direct(core_filter: str = "", rename: str | None = None):
    query = ""
    if "date." in core_filter:
        query += 'import "date"\n'
    if "interpolate." in core_filter:
        query += 'import "interpolate"\n'

    query += f"""
from(bucket: "existenzApi")
  |> range(start: {start.isoformat()}, stop: {end.isoformat()})
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
"""
    query += core_filter

    query += """
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement", "loc"])
"""

    df = store.query_raw(query)
    if rename:
        df = df.rename(columns={TEMP: rename})
    return df.drop(columns=["result", "table"])

In [ ]:
df_raw = get_direct(rename="raw")

In [ ]:
df_mean = get_direct(
    """
  |> aggregateWindow(every: 1h, fn: mean)
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
""",
    rename="mean",
)

df_mean

In [ ]:
df_first = get_direct(
    """
  |> aggregateWindow(every: 1h, fn: first, timeSrc: "_start")
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
""",
    rename="first",
)

In [ ]:
df_last = get_direct(
    """
  |> aggregateWindow(every: 1h, fn: last)
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
""",
    rename="last",
)

In [ ]:
df_exact = get_direct(
    """
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
""",
    rename="exact",
)

In [ ]:
df_interp = get_direct(
    """
  |> interpolate.linear(every: 10m)
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
""",
    rename="interp",
)

In [ ]:
df = join_many(df_raw, df_mean, df_first, df_last, df_exact, df_interp, on=TIME)
df

In [ ]:
df.describe()

In [ ]:
px.scatter(df, x=TIME, y=[c for c in df.columns if c != TIME])

In [ ]:
simp = df[[TIME, "first", "interp"]].dropna()
simp["diff"] = simp["first"] - simp["interp"]
simp["abs_diff"] = simp["diff"].abs()

In [ ]:
px.scatter(simp, TIME, "diff")

In [ ]:
percentiles = [0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
# if we think the large differences are concentrated in some months in some hours
# we could pessimistically estimate that during 2 months, 1.44h of the day we are off by the 99% quantile or more.
# if all is concentrated in 1 month of the year, just 17min per day (on avg) would be off be the 99.9% quantile or more.

In [ ]:
simp = simp[simp["abs_diff"] < 1]
# if the diff is 1° or more we're dealing with some heavy outliers,
# since even the temperature itself reeallly rarely does such changes within an hour.
simp.describe(percentiles=percentiles)

In [ ]:
diffs = df[[TIME]].copy()
for c in ["mean", "first", "last", "exact"]:
    diffs[c] = (df[c] - df["interp"]).abs()

diffs.describe(percentiles=percentiles)

## Decision on target aggregation

For the usecase at aare.guru predicting the hourly mean is wrong. Instead, we should work with the exact
temperature at the moment. Taking only data at full hours ('exact' above) doesn't work well because of inconsistencies
in upstream data like irregular intervals and short outages.
We therefore assume that interpolation on the 10min data and then subsamplig/picking only the data points at the full hours is
the best possible way to select **current, unmodified data at the current time**.
Since interpolation is expensive and would need some extra code to make it work, we want to use a simpler but still representative aggregation.

This analysis shows that **first** is the closest by far. It also shows that the hourly mean method is on average 0.05-0.075 °C off of the desired target.
This may not seem like much but **75% of all summer hours see an increase of 0.5°C or more** (see below). On average, the hourly increase is 0.15, so being off half of that
shifts the entire series by roughly 30min, which also confirms what we expected (hourly means should approximate the temperature at the 30min mark given that the data is often
monotonous and approximately linear within an hour).

In [ ]:
temp_diffs = df[[TIME, "first"]].set_index(TIME).resample("1h").first().diff().abs().reset_index()
temp_diffs = temp_diffs[temp_diffs["first"] < 2]  # super lenient outlier removal
months = temp_diffs[TIME].dt.month
temp_diffs = temp_diffs[((months >= 4) & (months <= 9))]
temp_diffs.describe(percentiles=percentiles)

In [ ]:
np.searchsorted(np.sort(temp_diffs["first"].values), 0.05) / len(temp_diffs)

In [ ]:
x = get_direct(
    """
  |> aggregateWindow(every: 1h, fn: first, timeSrc: "_start")
""",
    rename="first",
)
x

In [ ]:
resample(x)

Also tested to see and make sure `resample` correctly fixes the weird first or last data point that matches
the start or stop of the query range. As a nice side effect, `timeSrc: "_start"` has this weird point in
the beginning instead of the end which is cut off anyway in the service.

### todo

- analyze these different queries

```
import "date"

from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> aggregateWindow(every: 1h, fn: first, timeSrc: "_start")
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])


import "date"

from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])


from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])
```


- maybe do some tests with interpolate.linear, but the problem is, we only want to interpolate in max 1h gaps.
  Could in theory do 2 queries, one with interpolate and then minute == 0 and another without interpolate but agg(first).
  Then use interpolated ones except for places where agg(first) is none (no value in the entire hour).
  Slightly more expensive but it's kinda fancy and I think it might make it more robust (but more magic-y).
  In a first fix, just using agg(first) would probably suffice and is much simpler.


```
import "interpolate"
import "date"

from(bucket: "existenzApi")
  |> range(start: v.timeRangeStart, stop: v.timeRangeStop)
  |> filter(fn: (r) => r["_measurement"] == "hydro")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> filter(fn: (r) => r["loc"] == "2135")
  |> interpolate.linear(every: 10m)
  |> filter(fn: (r) => date.minute(t: r._time) == 0)
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> drop(columns: ["_start", "result", "_stop", "table", "_measurement"])
  |> rename(columns: {"_time": "time", "temperature": "Wassertemperatur Bern"})
  |> keep(columns: ["time", "Wassertemperatur Bern"])
```


- implement "first" as special case with timeSrc
- implement last, mean, etc. as catchall agg without timeSrc
- implement "exact" with minute == 0
- make script to do historical forecasting on data from postgres db. doesn't have to be pretty yet, just for comparison.
- make tool to show multiple different production forecast alongside 10min measurements for comparison.
- implement bias correction based on weighted linear extrapolation of the last 3-4 10min datapoints up to the full hour we predicted.
  maybe give claude a shot at this; it's simple to describe, but maybe not so simple to implement. testing is a challenge.
- (interactive) tuning of the bias correction parameters using the historical forecasts from the prod db
- refresh oraku2db mirrors to represent the target better (same target as model is trained on: first or if there's time fancy interpolate)
- maybe investigate the stupid cases where the model wrongly predicts down like 23.04.2026. I assume it's because it "think" that we're
  in a different phase because there was some unusual fluctuation, e.g. increase at 03:00 or in the case of 23.04.2026, it increased from 02:00 up to 06:00
  and then back down until 09:00 where it started to rise sharply. It should put more focus on the most recent hours.
